# Simulation Studies

This notebook walks through four representative simulation examples so readers can see how the MetVAE workflow behaves under different data-generating settings: with or without confounders, and with or without zero inflation.

The repeated simulation code is organized into a few helper functions first, and then reused across the four example scenarios.


In [1]:
import numpy as np
import pandas as pd

from metvae.model import MetVAE
from metvae.sim import sim_data


## 1. Helper functions used throughout the notebook

The next cell collects the repeated logic for simulation, model fitting, and performance evaluation. Defining these helpers once makes the example sections shorter and helps readers focus on what changes from one scenario to the next.

In [2]:
def simulate_metabolomics_dataset(n, d, zero_prop, seed, with_confounders):
    cor_pairs = int(0.2 * d)
    mu = list(range(10, 15))
    da_prop = 0.1

    np.random.seed(seed)

    smd = None
    sim_kwargs = {
        'n': n,
        'd': d,
        'cor_pairs': cor_pairs,
        'mu': mu,
        'da_prop': da_prop,
    }

    if with_confounders:
        smd = pd.DataFrame(
            {
                'x1': np.random.randn(n),
                'x2': np.random.choice(['a', 'b'], size=n, replace=True),
            },
            index=[f's{i}' for i in range(n)],
        )
        sim_kwargs.update({'x': smd, 'cont_list': ['x1'], 'cat_list': ['x2']})

    sim = sim_data(**sim_kwargs)
    y = sim['y']
    true_cor = sim['cor_matrix']

    log_y = np.log(y)
    log_sample_bias = np.log(np.random.uniform(1e-3, 1e-1, size=n))
    log_feature_bias = np.log(np.random.uniform(1e-1, 1, size=d))
    log_data = log_y + log_sample_bias[:, np.newaxis]
    log_data = log_data + log_feature_bias.reshape(1, d)
    data = np.exp(log_data)

    thresholds = np.quantile(data, zero_prop, axis=0)
    data_miss = np.where(data < thresholds, 0, data)
    data_miss = pd.DataFrame(data_miss, index=y.index, columns=y.columns)

    return {
        'data': data_miss,
        'meta': smd,
        'true_cor': true_cor,
        'sim': sim,
    }

def fit_metvae_model(data, meta, n, d):
    model_kwargs = {
        'data': data,
        'features_as_rows': False,
        'meta': meta,
        'continuous_covariate_keys': ['x1'] if meta is not None else None,
        'categorical_covariate_keys': ['x2'] if meta is not None else None,
        'latent_dim': min(n, d),
    }

    if meta is None:
        model_kwargs['feature_zero_threshold'] = None

    model = MetVAE(**model_kwargs)
    model.train(
        batch_size=100,
        num_workers=0,
        max_epochs=1000,
        learning_rate=1e-2,
    )
    return model

def summarize_sparse_estimate(est_cor, true_cor):
    est_values = est_cor.values if isinstance(est_cor, pd.DataFrame) else est_cor
    true_idx = true_cor[np.tril_indices_from(true_cor, k=-1)] != 0
    est_idx = est_values[np.tril_indices_from(est_values, k=-1)] != 0
    tpr = np.sum(est_idx & true_idx) / np.sum(true_idx)
    fpr = np.sum(est_idx & ~true_idx) / np.sum(~true_idx)
    fdr = np.sum(est_idx & ~true_idx) / np.sum(est_idx)
    return {'tpr': tpr, 'fpr': fpr, 'fdr': fdr}

def print_metrics(label, metrics, extra=None):
    print(f"{label}: tpr = {metrics['tpr']}, fpr = {metrics['fpr']}, fdr = {metrics['fdr']}")
    if extra is not None:
        print(extra)

def run_pvalue_pipeline(model, true_cor, cutoff=0.05):
    model.get_corr(num_sim=100)
    results_metvae = model.sparse_by_p(p_adj_method='fdr_bh', cutoff=cutoff)
    metrics = summarize_sparse_estimate(results_metvae['sparse_estimate'], true_cor)
    return results_metvae, metrics

def run_sec_fixed_rho(model, true_cor, rho=2.2):
    results_metvae = model.sparse_by_sec(rho=rho)
    metrics = summarize_sparse_estimate(results_metvae['sparse_estimate'], true_cor)
    return results_metvae, metrics

def run_sec_grid_search(model, true_cor, c_grid=tuple(float(x) for x in range(8, 16))):
    results_metvae = model.sparse_by_sec(c_grid=c_grid)
    metrics = summarize_sparse_estimate(results_metvae['sparse_estimate'], true_cor)
    return results_metvae, metrics

## 2. Four representative simulation examples

The examples below all use the same basic simulation workflow, but vary along two axes:

- whether confounders are present, and
- whether zero inflation is present.

Within each example, we fit MetVAE once and then compare two sparsification strategies:

- p-value filtering, and
- SEC, shown with both a fixed `rho` and a grid-based search.

### Example 1: no confounders, with zeros

This first example is the simplest sparse setting: no covariates are adjusted for, but the observed abundance table contains zeros introduced by censoring.

In [3]:
n, d, zero_prop, seed = 100, 500, 0.3, 24

example_1 = simulate_metabolomics_dataset(
    n=n,
    d=d,
    zero_prop=zero_prop,
    seed=seed,
    with_confounders=False,
)

model = fit_metvae_model(example_1['data'], example_1['meta'], n=n, d=d)
true_cor = example_1['true_cor']

Start: samples=100, features=500
Filtered samples: none (no sample_zero_threshold provided)
After zero filtering: samples=100, features=500
Post-cleaning (convert negative values to zeros and drop all-zero samples): samples=100, features=500


  0%|          | 0/1000 [00:00<?, ?it/s]

  2%|▎         | 25/1000 [00:00<00:03, 247.74it/s]

  5%|▌         | 54/1000 [00:00<00:03, 270.98it/s]

  8%|▊         | 82/1000 [00:00<00:03, 267.11it/s]

...

 91%|█████████ | 909/1000 [00:03<00:00, 294.11it/s]

 94%|█████████▍| 939/1000 [00:03<00:00, 293.22it/s]

 97%|█████████▋| 969/1000 [00:03<00:00, 291.55it/s]

100%|█████████▉| 999/1000 [00:03<00:00, 286.76it/s]

100%|██████████| 1000/1000 [00:03<00:00, 287.19it/s]

#### P-value filtering

We first estimate the correlation matrix through repeated imputations and then apply the p-value-based sparsification rule.

In [4]:
results_metvae, metrics = run_pvalue_pipeline(model, true_cor)
print_metrics('P-value filtering', metrics)

P-value filtering: tpr = 0.92, fpr = 5.615724027276374e-05, fdr = 0.0707070707070707


#### SEC

We then evaluate SEC twice: once with a fixed `rho` and once with a small grid search that reports the selected value.

In [5]:
results_metvae, metrics = run_sec_fixed_rho(model, true_cor, rho=2.2)
print_metrics('SEC (fixed rho)', metrics)

SEC (fixed rho): tpr = 0.9, fpr = 4.0112314480545525e-05, fdr = 0.05263157894736842


In [6]:
results_metvae, metrics = run_sec_grid_search(model, true_cor, c_grid=tuple(float(x) for x in range(8, 16)))
print_metrics('SEC (grid search)', metrics, extra=f"rho = {results_metvae['best_rho']}")

SEC (grid search): tpr = 0.85, fpr = 0.0, fdr = 0.0
rho = 2.7699017450199266


### Example 2: no confounders, without zeros

This example isolates the effect of removing zero inflation while keeping the rest of the setup unchanged.

In [7]:
n, d, zero_prop, seed = 100, 500, 0.0, 44

example_2 = simulate_metabolomics_dataset(
    n=n,
    d=d,
    zero_prop=zero_prop,
    seed=seed,
    with_confounders=False,
)

model = fit_metvae_model(example_2['data'], example_2['meta'], n=n, d=d)
true_cor = example_2['true_cor']

Start: samples=100, features=500
Filtered samples: none (no sample_zero_threshold provided)
After zero filtering: samples=100, features=500
Post-cleaning (convert negative values to zeros and drop all-zero samples): samples=100, features=500


  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 4/1000 [00:00<00:33, 29.70it/s]

  1%|          | 7/1000 [00:00<00:34, 28.95it/s]

  4%|▍         | 43/1000 [00:00<00:05, 166.61it/s]

...

 86%|████████▋ | 863/1000 [00:02<00:00, 391.60it/s]

 90%|█████████ | 903/1000 [00:02<00:00, 382.80it/s]

 94%|█████████▍| 942/1000 [00:02<00:00, 377.71it/s]

 98%|█████████▊| 980/1000 [00:02<00:00, 375.71it/s]

100%|██████████| 1000/1000 [00:02<00:00, 347.47it/s]

#### P-value filtering

We reuse the same evaluation sequence so the only change across examples is the data-generating setting.

In [8]:
results_metvae, metrics = run_pvalue_pipeline(model, true_cor)
print_metrics('P-value filtering', metrics)

P-value filtering: tpr = 0.94, fpr = 3.208985158443642e-05, fdr = 0.04081632653061224


#### SEC

In [9]:
results_metvae, metrics = run_sec_fixed_rho(model, true_cor, rho=2.2)
print_metrics('SEC (fixed rho)', metrics)

SEC (fixed rho): tpr = 0.94, fpr = 2.4067388688327318e-05, fdr = 0.030927835051546393


In [10]:
results_metvae, metrics = run_sec_grid_search(model, true_cor, c_grid=tuple(float(x) for x in range(8, 16)))
print_metrics('SEC (grid search)', metrics, extra=f"rho = {results_metvae['best_rho']}")

SEC (grid search): tpr = 0.89, fpr = 0.0, fdr = 0.0
rho = 2.603707640318731


### Example 3: with confounders, with zeros

We now move to the covariate-adjusted setting. The simulated metadata include one continuous and one categorical confounder, and zeros are again introduced into the abundance matrix.

In [11]:
n, d, zero_prop, seed = 100, 500, 0.3, 42

example_3 = simulate_metabolomics_dataset(
    n=n,
    d=d,
    zero_prop=zero_prop,
    seed=seed,
    with_confounders=True,
)

model = fit_metvae_model(example_3['data'], example_3['meta'], n=n, d=d)
true_cor = example_3['true_cor']

Start: samples=100, features=500
Filtered features: removed 0 (threshold 0.30)
Filtered samples: none (no sample_zero_threshold provided)
After zero filtering: samples=100, features=500
Post-cleaning (convert negative values to zeros and drop all-zero samples): samples=100, features=500


  0%|          | 0/1000 [00:00<?, ?it/s]

  2%|▏         | 21/1000 [00:00<00:04, 202.74it/s]

  4%|▍         | 45/1000 [00:00<00:04, 223.48it/s]

  7%|▋         | 70/1000 [00:00<00:03, 235.09it/s]

...

 92%|█████████▏| 924/1000 [00:03<00:00, 214.88it/s]

 95%|█████████▍| 946/1000 [00:04<00:00, 207.94it/s]

 97%|█████████▋| 967/1000 [00:04<00:00, 204.15it/s]

 99%|█████████▉| 990/1000 [00:04<00:00, 208.98it/s]

100%|██████████| 1000/1000 [00:04<00:00, 229.84it/s]

#### P-value filtering

In [12]:
results_metvae, metrics = run_pvalue_pipeline(model, true_cor)
print_metrics('P-value filtering', metrics)

P-value filtering: tpr = 0.73, fpr = 0.00012835940633774569, fdr = 0.1797752808988764


#### SEC

In [13]:
results_metvae, metrics = run_sec_fixed_rho(model, true_cor, rho=2.2)
print_metrics('SEC (fixed rho)', metrics)

SEC (fixed rho): tpr = 0.72, fpr = 0.00011231448054552748, fdr = 0.16279069767441862


In [14]:
results_metvae, metrics = run_sec_grid_search(model, true_cor, c_grid=tuple(float(x) for x in range(8, 16)))
print_metrics('SEC (grid search)', metrics, extra=f"rho = {results_metvae['best_rho']}")

SEC (grid search): tpr = 0.5, fpr = 0.0, fdr = 0.0
rho = 2.9083968322709226


### Example 4: with confounders, without zeros

The final example removes zero inflation from the confounded setting so readers can compare all four combinations side by side.

In [15]:
n, d, zero_prop, seed = 100, 500, 0.0, 92

example_4 = simulate_metabolomics_dataset(
    n=n,
    d=d,
    zero_prop=zero_prop,
    seed=seed,
    with_confounders=True,
)

model = fit_metvae_model(example_4['data'], example_4['meta'], n=n, d=d)
true_cor = example_4['true_cor']

Start: samples=100, features=500
Filtered features: removed 0 (threshold 0.30)
Filtered samples: none (no sample_zero_threshold provided)
After zero filtering: samples=100, features=500
Post-cleaning (convert negative values to zeros and drop all-zero samples): samples=100, features=500


  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 3/1000 [00:00<00:58, 17.00it/s]

  2%|▏         | 21/1000 [00:00<00:10, 89.80it/s]

  6%|▌         | 58/1000 [00:00<00:04, 197.71it/s]

...

 88%|████████▊ | 882/1000 [00:02<00:00, 339.83it/s]

 92%|█████████▏| 916/1000 [00:02<00:00, 338.59it/s]

 95%|█████████▌| 952/1000 [00:03<00:00, 343.97it/s]

 99%|█████████▉| 988/1000 [00:03<00:00, 346.30it/s]

100%|██████████| 1000/1000 [00:03<00:00, 313.33it/s]

#### P-value filtering

In [16]:
results_metvae, metrics = run_pvalue_pipeline(model, true_cor)
print_metrics('P-value filtering', metrics)

P-value filtering: tpr = 0.98, fpr = 3.208985158443642e-05, fdr = 0.0392156862745098


#### SEC

In [17]:
results_metvae, metrics = run_sec_fixed_rho(model, true_cor, rho=2.2)
print_metrics('SEC (fixed rho)', metrics)

SEC (fixed rho): tpr = 0.98, fpr = 1.604492579221821e-05, fdr = 0.02


In [18]:
results_metvae, metrics = run_sec_grid_search(model, true_cor, c_grid=tuple(float(x) for x in range(8, 16)))
print_metrics('SEC (grid search)', metrics, extra=f"rho = {results_metvae['best_rho']}")

SEC (grid search): tpr = 0.94, fpr = 8.022462896109105e-06, fdr = 0.010526315789473684
rho = 2.520610587968133
